# Data preparation

Aim: data frame containing the percentage of lexical shibboleths, other shibboleths, non shibboleths and if applicable other annotations

## Helper functions

In [ ]:
%pip install pandas seaborn matplotlib

In [1]:
import pandas as pd
import os, re, sys
import numpy as np

### List-to-method correspondences

In [2]:
def get_method_dict(file):
    method_dict = {'average' : 'average'}
    with open(file, 'r') as f:
        lines = [line.rstrip() for line in f.readlines()]
        i = 1
        for l in lines:
            match = re.search(r'^.+attrib=([^_]+)_mpd.+_thresh=([^_]+)_.+$', l)
            method = match.group(1)
            threshold = match.group(2)
            key = 'list' + str(i)
            method_dict[key] = method + '_' + threshold
            i += 1
    return method_dict


Example:

In [3]:
bcms_setimes_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.bcms_setimes.txt')
bcms_setimes_methods_df = pd.DataFrame.from_dict(bcms_setimes_methods, orient='index', columns=['method'])
bcms_setimes_methods_df

,method
average,average
list1,loo_0.6
list2,loo_0.1
list3,shap_0.6
list4,shap_0.1
list5,ig_0.1
list6,lime_0.6
list7,lime_0.1
list8,ig_0.6


### Importing all lists and labelling them with list ID

In [4]:
def make_dataset_df(folder):
    dfs = list()
    files = os.listdir(folder)
    for file in files:
        number = re.search(r'\d', file)
        if number:
            id = number.group()
        else:
            print("Skip file:", file)
            continue
        if file.endswith('.csv'):
            df = pd.read_csv(folder + '/' + file)
            df['file'] = "list" + id
            dfs.append(df)
        elif file.endswith('.txt'):
            df = pd.read_csv(folder + '/' + file, sep='\t')
            df['file'] = "list" + id
            dfs.append(df)
    full_df = pd.concat(dfs, ignore_index=True)
    return full_df

Example:

In [5]:
bcms_twitter_df = make_dataset_df('../../manual_eval/bcms_twitter')
bcms_twitter_df

,token,label,score,selection_freq,in_blacklist,shibboleth,feature,feature2,note,file
0,😎🌞<unk>☕️🍉,bs,1.525842,0.104167,NaN,no,emoji,NaN,NaN,list6
1,☕️😎,bs,1.430635,0.104167,NaN,no,emoji,NaN,NaN,list6
2,unk>😏,bs,1.371145,0.104167,NaN,no,emoji,NaN,NaN,list6
3,🙈<unk>🙊,bs,1.352293,0.104167,NaN,no,emoji,NaN,NaN,list6
4,👍😏,bs,1.306748,0.104167,NaN,no,emoji,NaN,NaN,list6
...,...,...,...,...,...,...,...,...,...,...
3195,neizvesno,sr,0.611152,0.114583,NaN,yes,phon,NaN,NaN,list8
3196,osetiti,sr,0.610422,0.187500,NaN,yes,phon,NaN,NaN,list8
3197,pobedama,sr,0.608175,0.104167,NaN,yes,phon,NaN,NaN,list8
3198,miholjskoleto,sr,0.608170,0.104167,NaN,yes,phon,nonminimal,NaN,list8


### Getting the number of annotations per list (for proper percentage normalization)

In [6]:
def get_number_annots(df):
    number_annots = df.value_counts('file').to_frame()
    number_annots = number_annots.rename(columns={number_annots.columns[0]: "total"})
    return number_annots

Example:

In [7]:
bcms_twitter_nannots = get_number_annots(bcms_twitter_df)
bcms_twitter_nannots

,total
file,
list6,400
list7,400
list5,400
list4,400
list1,400
list3,400
list2,400
list8,400


In [ ]:
bcms_setimes_methods = get_method_dict('randomized_lists/randomized_lists.bcms_setimes.txt')
bcms_twitter_methods = get_method_dict('randomized_lists/randomized_lists.bcms_twitter.txt')


german_jodel_methods = get_method_dict('randomized_lists/randomized_lists.jodel.txt')



In [ ]:
bcms_setimes_df = make_dataset_df('bcms_setimes')
bcms_setimes_df

## Apply the pipeline to all language groups

### bcms_setimes

In [8]:
bcms_setimes_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.bcms_setimes.txt')
bcms_setimes_methods_df = pd.DataFrame.from_dict(bcms_setimes_methods, orient='index', columns=['method'])
bcms_setimes_methods_df

,method
average,average
list1,loo_0.6
list2,loo_0.1
list3,shap_0.6
list4,shap_0.1
list5,ig_0.1
list6,lime_0.6
list7,lime_0.1
list8,ig_0.6


In [9]:
bcms_setimes_df = make_dataset_df('../../manual_eval/bcms_setimes')
bcms_setimes_df

Skip file: evaluateLists.py


,token,label,score,selection_freq,in_blacklist,shibboleth,feature,feature2,note,file
0,natjecateljskom,hr,0.241601,0.70,True,yes,lex,NaN,NaN,list6
1,liječniku,hr,0.232991,0.67,True,yes,lex,NaN,jek,list6
2,slijedili,hr,0.228810,0.69,True,no,phon,NaN,jek,list6
3,natjecalo,hr,0.221181,0.66,True,yes,lex,NaN,NaN,list6
4,strijelac,hr,0.211338,0.87,True,no,phon,NaN,jek,list6
...,...,...,...,...,...,...,...,...,...,...
1595,predsedavati,sr,0.362739,0.65,False,yes,phon,NaN,NaN,list8
1596,saopštavajući,sr,0.362684,0.92,False,yes,phon,NaN,NaN,list8
1597,evroatlantske,sr,0.362167,1.00,False,yes,phon,nonminimal,NaN,list8
1598,ignorisati,sr,0.361948,0.62,False,yes,morph-d,NaN,NaN,list8


In [10]:
bcms_setimes_nannots = get_number_annots(bcms_setimes_df)
bcms_setimes_nannots

,total
file,
list6,200
list7,200
list5,200
list4,200
list1,200
list3,200
list2,200
list8,200


In [11]:
bcms_setimes_shib_distro = bcms_setimes_df[(bcms_setimes_df['shibboleth'] == 'yes') & (bcms_setimes_df['feature'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
bcms_setimes_shib_distro['nonlex shib'] = bcms_setimes_df[(bcms_setimes_df['shibboleth'] == 'yes') & (bcms_setimes_df['feature'] != 'lex')].groupby('file').size()
bcms_setimes_shib_distro['non shib'] = bcms_setimes_df[bcms_setimes_df['shibboleth'] == 'no'].groupby('file').size()
bcms_setimes_shib_distro['rest'] = bcms_setimes_df[(bcms_setimes_df['shibboleth'] != 'no') & (bcms_setimes_df['shibboleth'] != 'yes')].groupby('file').size()
bcms_setimes_shib_distro = pd.concat([bcms_setimes_shib_distro, bcms_setimes_nannots], axis=1)
bcms_setimes_shib_distro['shibboleths: lexical'] = round(bcms_setimes_shib_distro['lex shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['shibboleths: other'] = round(bcms_setimes_shib_distro['nonlex shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['non shibboleths'] = round(bcms_setimes_shib_distro['non shib'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['other'] = round(bcms_setimes_shib_distro['rest'] / bcms_setimes_shib_distro['total'] * 100, 1)
bcms_setimes_shib_distro['perc sum'] = bcms_setimes_shib_distro['shibboleths: lexical'] + bcms_setimes_shib_distro['shibboleths: other'] \
+ bcms_setimes_shib_distro['non shibboleths'] + bcms_setimes_shib_distro['other']
bcms_setimes_shib_distro.loc['average'] = round(bcms_setimes_shib_distro.mean(axis=0), 1)
bcms_setimes_shib_distro

,lex shib,nonlex shib,non shib,rest,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc sum
file,,,,,,,,,,
list1,33.0,70.0,94.0,3.0,200.0,16.5,35.0,47.0,1.5,100.0
list2,14.0,49.0,131.0,6.0,200.0,7.0,24.5,65.5,3.0,100.0
list3,33.0,72.0,95.0,NaN,200.0,16.5,36.0,47.5,NaN,NaN
list4,15.0,45.0,135.0,5.0,200.0,7.5,22.5,67.5,2.5,100.0
list5,20.0,58.0,120.0,2.0,200.0,10.0,29.0,60.0,1.0,100.0
list6,37.0,85.0,75.0,3.0,200.0,18.5,42.5,37.5,1.5,100.0
list7,22.0,88.0,82.0,8.0,200.0,11.0,44.0,41.0,4.0,100.0
list8,37.0,96.0,67.0,NaN,200.0,18.5,48.0,33.5,NaN,NaN
average,26.4,70.4,99.9,4.5,200.0,13.2,35.2,49.9,2.2,100.0


In [12]:
bcms_setimes_shib_distro = pd.concat([bcms_setimes_shib_distro, bcms_setimes_methods_df], axis=1)
bcms_setimes_shib_distro


,lex shib,nonlex shib,non shib,rest,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc sum,method
list1,33.0,70.0,94.0,3.0,200.0,16.5,35.0,47.0,1.5,100.0,loo_0.6
list2,14.0,49.0,131.0,6.0,200.0,7.0,24.5,65.5,3.0,100.0,loo_0.1
list3,33.0,72.0,95.0,NaN,200.0,16.5,36.0,47.5,NaN,NaN,shap_0.6
list4,15.0,45.0,135.0,5.0,200.0,7.5,22.5,67.5,2.5,100.0,shap_0.1
list5,20.0,58.0,120.0,2.0,200.0,10.0,29.0,60.0,1.0,100.0,ig_0.1
list6,37.0,85.0,75.0,3.0,200.0,18.5,42.5,37.5,1.5,100.0,lime_0.6
list7,22.0,88.0,82.0,8.0,200.0,11.0,44.0,41.0,4.0,100.0,lime_0.1
list8,37.0,96.0,67.0,NaN,200.0,18.5,48.0,33.5,NaN,NaN,ig_0.6
average,26.4,70.4,99.9,4.5,200.0,13.2,35.2,49.9,2.2,100.0,average


In [13]:
bcms_setimes_shib_distro = bcms_setimes_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
bcms_setimes_shib_distro

,list,lex shib,nonlex shib,non shib,rest,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc sum
method,,,,,,,,,,,
average,average,26.4,70.4,99.9,4.5,200.0,13.2,35.2,49.9,2.2,100.0
ig_0.1,list5,20.0,58.0,120.0,2.0,200.0,10.0,29.0,60.0,1.0,100.0
ig_0.6,list8,37.0,96.0,67.0,NaN,200.0,18.5,48.0,33.5,NaN,NaN
lime_0.1,list7,22.0,88.0,82.0,8.0,200.0,11.0,44.0,41.0,4.0,100.0
lime_0.6,list6,37.0,85.0,75.0,3.0,200.0,18.5,42.5,37.5,1.5,100.0
loo_0.1,list2,14.0,49.0,131.0,6.0,200.0,7.0,24.5,65.5,3.0,100.0
loo_0.6,list1,33.0,70.0,94.0,3.0,200.0,16.5,35.0,47.0,1.5,100.0
shap_0.1,list4,15.0,45.0,135.0,5.0,200.0,7.5,22.5,67.5,2.5,100.0
shap_0.6,list3,33.0,72.0,95.0,NaN,200.0,16.5,36.0,47.5,NaN,NaN


In [14]:
bcms_setimes_overview = bcms_setimes_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
bcms_setimes_overview.to_csv('tables/bcms_setimes_overview.csv')
bcms_setimes_overview

,shibboleths: lexical,shibboleths: other,non shibboleths,other
method,,,,
average,13.2,35.2,49.9,2.2
ig_0.1,10.0,29.0,60.0,1.0
ig_0.6,18.5,48.0,33.5,NaN
lime_0.1,11.0,44.0,41.0,4.0
lime_0.6,18.5,42.5,37.5,1.5
loo_0.1,7.0,24.5,65.5,3.0
loo_0.6,16.5,35.0,47.0,1.5
shap_0.1,7.5,22.5,67.5,2.5
shap_0.6,16.5,36.0,47.5,NaN


### bcms_twitter

In [15]:
bcms_twitter_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.bcms_twitter.txt')
bcms_twitter_methods_df = pd.DataFrame.from_dict(bcms_twitter_methods, orient='index', columns=['method'])
bcms_twitter_methods_df

,method
average,average
list1,lime_0.6
list2,loo_0.1
list3,shap_0.1
list4,ig_0.6
list5,loo_0.6
list6,ig_0.1
list7,shap_0.6
list8,lime_0.1


In [16]:
bcms_twitter_df = make_dataset_df('../../manual_eval/bcms_twitter')
bcms_twitter_df

,token,label,score,selection_freq,in_blacklist,shibboleth,feature,feature2,note,file
0,😎🌞<unk>☕️🍉,bs,1.525842,0.104167,NaN,no,emoji,NaN,NaN,list6
1,☕️😎,bs,1.430635,0.104167,NaN,no,emoji,NaN,NaN,list6
2,unk>😏,bs,1.371145,0.104167,NaN,no,emoji,NaN,NaN,list6
3,🙈<unk>🙊,bs,1.352293,0.104167,NaN,no,emoji,NaN,NaN,list6
4,👍😏,bs,1.306748,0.104167,NaN,no,emoji,NaN,NaN,list6
...,...,...,...,...,...,...,...,...,...,...
3195,neizvesno,sr,0.611152,0.114583,NaN,yes,phon,NaN,NaN,list8
3196,osetiti,sr,0.610422,0.187500,NaN,yes,phon,NaN,NaN,list8
3197,pobedama,sr,0.608175,0.104167,NaN,yes,phon,NaN,NaN,list8
3198,miholjskoleto,sr,0.608170,0.104167,NaN,yes,phon,nonminimal,NaN,list8


In [17]:
bcms_twitter_nannots = get_number_annots(bcms_twitter_df)
bcms_twitter_nannots

,total
file,
list6,400
list7,400
list5,400
list4,400
list1,400
list3,400
list2,400
list8,400


In [18]:
bcms_twitter_shib_distro = bcms_twitter_df[(bcms_twitter_df['shibboleth'] == 'yes') & (bcms_twitter_df['feature'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
bcms_twitter_shib_distro['nonlex shib'] = bcms_twitter_df[(bcms_twitter_df['shibboleth'] == 'yes') & (bcms_twitter_df['feature'] != 'lex')].groupby('file').size()
bcms_twitter_shib_distro['non shib'] = bcms_twitter_df[bcms_twitter_df['shibboleth'] == 'no'].groupby('file').size()
bcms_twitter_shib_distro['rest'] = bcms_twitter_df[(bcms_twitter_df['shibboleth'] != 'no') & (bcms_twitter_df['shibboleth'] != 'yes')].groupby('file').size()
bcms_twitter_shib_distro = pd.concat([bcms_twitter_shib_distro, bcms_twitter_nannots], axis=1)
bcms_twitter_shib_distro['shibboleths: lexical'] = round(bcms_twitter_shib_distro['lex shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['shibboleths: other'] = round(bcms_twitter_shib_distro['nonlex shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['non shibboleths'] = round(bcms_twitter_shib_distro['non shib'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['other'] = round(bcms_twitter_shib_distro['rest'] / bcms_twitter_shib_distro['total'] * 100, 1)
bcms_twitter_shib_distro['perc sum'] = bcms_twitter_shib_distro['shibboleths: lexical'] + bcms_twitter_shib_distro['shibboleths: other'] \
+ bcms_twitter_shib_distro['non shibboleths'] + bcms_twitter_shib_distro['other']
bcms_twitter_shib_distro.loc['average'] = round(bcms_twitter_shib_distro.mean(axis=0), 1)
bcms_twitter_shib_distro

,lex shib,nonlex shib,non shib,rest,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc sum
file,,,,,,,,,,
list1,15.0,109.0,268.0,8.0,400.0,3.8,27.3,67.0,2.0,100.1
list2,13.0,78.0,302.0,7.0,400.0,3.2,19.5,75.5,1.8,100.0
list3,19.0,100.0,273.0,8.0,400.0,4.8,25.0,68.2,2.0,100.0
list4,9.0,108.0,269.0,14.0,400.0,2.2,27.0,67.2,3.5,99.9
list5,9.0,97.0,284.0,10.0,400.0,2.2,24.2,71.0,2.5,99.9
list6,16.0,79.0,293.0,12.0,400.0,4.0,19.8,73.2,3.0,100.0
list7,16.0,112.0,262.0,10.0,400.0,4.0,28.0,65.5,2.5,100.0
list8,21.0,113.0,259.0,7.0,400.0,5.2,28.2,64.8,1.8,100.0
average,14.8,99.5,276.2,9.5,400.0,3.7,24.9,69.0,2.4,100.0


In [19]:
bcms_twitter_shib_distro = pd.concat([bcms_twitter_shib_distro, bcms_twitter_methods_df], axis = 1)
bcms_twitter_shib_distro

,lex shib,nonlex shib,non shib,rest,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc sum,method
list1,15.0,109.0,268.0,8.0,400.0,3.8,27.3,67.0,2.0,100.1,lime_0.6
list2,13.0,78.0,302.0,7.0,400.0,3.2,19.5,75.5,1.8,100.0,loo_0.1
list3,19.0,100.0,273.0,8.0,400.0,4.8,25.0,68.2,2.0,100.0,shap_0.1
list4,9.0,108.0,269.0,14.0,400.0,2.2,27.0,67.2,3.5,99.9,ig_0.6
list5,9.0,97.0,284.0,10.0,400.0,2.2,24.2,71.0,2.5,99.9,loo_0.6
list6,16.0,79.0,293.0,12.0,400.0,4.0,19.8,73.2,3.0,100.0,ig_0.1
list7,16.0,112.0,262.0,10.0,400.0,4.0,28.0,65.5,2.5,100.0,shap_0.6
list8,21.0,113.0,259.0,7.0,400.0,5.2,28.2,64.8,1.8,100.0,lime_0.1
average,14.8,99.5,276.2,9.5,400.0,3.7,24.9,69.0,2.4,100.0,average


In [20]:
bcms_twitter_shib_distro  = bcms_twitter_shib_distro.sort_values(by='method').reset_index(names=['list']).set_index('method')
bcms_twitter_shib_distro

,list,lex shib,nonlex shib,non shib,rest,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc sum
method,,,,,,,,,,,
average,average,14.8,99.5,276.2,9.5,400.0,3.7,24.9,69.0,2.4,100.0
ig_0.1,list6,16.0,79.0,293.0,12.0,400.0,4.0,19.8,73.2,3.0,100.0
ig_0.6,list4,9.0,108.0,269.0,14.0,400.0,2.2,27.0,67.2,3.5,99.9
lime_0.1,list8,21.0,113.0,259.0,7.0,400.0,5.2,28.2,64.8,1.8,100.0
lime_0.6,list1,15.0,109.0,268.0,8.0,400.0,3.8,27.3,67.0,2.0,100.1
loo_0.1,list2,13.0,78.0,302.0,7.0,400.0,3.2,19.5,75.5,1.8,100.0
loo_0.6,list5,9.0,97.0,284.0,10.0,400.0,2.2,24.2,71.0,2.5,99.9
shap_0.1,list3,19.0,100.0,273.0,8.0,400.0,4.8,25.0,68.2,2.0,100.0
shap_0.6,list7,16.0,112.0,262.0,10.0,400.0,4.0,28.0,65.5,2.5,100.0


In [21]:
bcms_twitter_overview = bcms_twitter_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
bcms_twitter_overview.to_csv('tables/bcms_twitter_overview.csv')
bcms_twitter_overview

,shibboleths: lexical,shibboleths: other,non shibboleths,other
method,,,,
average,3.7,24.9,69.0,2.4
ig_0.1,4.0,19.8,73.2,3.0
ig_0.6,2.2,27.0,67.2,3.5
lime_0.1,5.2,28.2,64.8,1.8
lime_0.6,3.8,27.3,67.0,2.0
loo_0.1,3.2,19.5,75.5,1.8
loo_0.6,2.2,24.2,71.0,2.5
shap_0.1,4.8,25.0,68.2,2.0
shap_0.6,4.0,28.0,65.5,2.5


### german_jodel

In [22]:
german_jodel_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.jodel.txt')
german_jodel_methods_df = pd.DataFrame.from_dict(german_jodel_methods, orient='index', columns=['method'])
german_jodel_methods_df

,method
average,average
list1,ig_0.1
list2,ig_0.6
list3,lime_0.1
list4,lime_0.6
list5,shap_0.6
list6,shap_0.1
list7,loo_0.1
list8,loo_0.6


In [23]:
german_jodel_df = make_dataset_df('../../manual_eval/german_jodel')
german_jodel_df

Skip file: readme.txt


,token,label,score,selection_freq,code,other_variety,note,file
0,klagenfurt?🙈,AT,0.784082,0.10,NE,NaN,NaN,list6
1,flachwitzfreitag,AT,0.720582,0.98,unmarked,NaN,NaN,list6
2,salzburgring,AT,0.698946,0.10,NE,NaN,NaN,list6
3,österreichsollösterreichbleiben,AT,0.678802,0.10,NE,NaN,NaN,list6
4,körndln,AT,0.676768,0.10,morph,NaN,NaN,list6
...,...,...,...,...,...,...,...,...
3195,heinerfest,DE_S,0.313043,0.87,NE,NaN,NaN,list8
3196,ilmenau,DE_S,0.312112,1.00,NE,NaN,NaN,list8
3197,heimkommen,DE_S,0.312046,0.74,unmarked,NaN,NaN,list8
3198,passau,DE_S,0.312020,0.97,NE,NaN,NaN,list8


In [24]:
german_jodel_df['other_variety'].value_counts()

other_variety
yes    10
Yes     3
Name: count, dtype: int64

In [26]:
german_jodel_df.value_counts('code')

code
NE          1982
unmarked     401
phon         346
lex          278
morph        187
spell          5
other          1
Name: count, dtype: int64

In [25]:
german_jodel_nannots = get_number_annots(german_jodel_df)
german_jodel_nannots

,total
file,
list6,400
list7,400
list5,400
list4,400
list1,400
list3,400
list2,400
list8,400


In [27]:
# Named entities are not shibboleths
# Only single-variety shibboleths count
german_jodel_df['shibboleth'] = 'no'
german_jodel_df.loc[(german_jodel_df['code'] != 'NE') & (german_jodel_df['code'] != 'unmarked') \
& (german_jodel_df['other_variety'].isna()) & (german_jodel_df['note'].isna()) , 'shibboleth'] = 'yes'
# sanity check:
german_jodel_df[(german_jodel_df['code'] == 'lex') & (german_jodel_df['shibboleth'] == 'no')]

,token,label,score,selection_freq,code,other_variety,note,file,shibboleth
469,gschwind,AT,0.439367,0.22,lex,NaN,not exclusive,list7,no
840,ausschaun,AT,0.238637,0.60,lex,NaN,not exclusive,list5,no
856,amal,AT,0.195144,0.99,lex,NaN,not exclusive,list5,no
857,schaun,AT,0.194866,1.00,lex,NaN,not exclusive,list5,no
868,schaut,AT,0.158346,1.00,lex,NaN,not exclusive,list5,no
869,heit,AT,0.155996,1.00,lex,NaN,not exclusive,list5,no
873,geschaut,AT,0.151669,0.75,lex,NaN,not exclusive,list5,no
1242,amal,AT,0.274777,1.00,lex,NaN,not exclusive,list4,no
1256,gscheit,AT,0.241944,0.63,lex,NaN,not exclusive,list4,no
1276,matura,AT,0.203438,0.85,lex,NaN,not exclusive,list4,no


In [28]:
german_jodel_shib_distro = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] == 'lex')].groupby('file').size().to_frame(name='lex shib')
german_jodel_shib_distro['nonlex shib'] = german_jodel_df[(german_jodel_df['shibboleth'] == 'yes') & (german_jodel_df['code'] != 'lex')].groupby('file').size()
german_jodel_shib_distro['non shib'] = german_jodel_df[german_jodel_df['shibboleth'] == 'no'].groupby('file').size()
german_jodel_shib_distro.loc[:,'sum'] = german_jodel_shib_distro.sum(axis=1)
german_jodel_shib_distro = pd.concat([german_jodel_shib_distro, german_jodel_nannots], axis=1)
german_jodel_shib_distro['shibboleths: lexical'] = round(german_jodel_shib_distro['lex shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['shibboleths: other'] = round(german_jodel_shib_distro['nonlex shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['non shibboleths'] = round(german_jodel_shib_distro['non shib'] / german_jodel_shib_distro['total'] * 100, 1)
german_jodel_shib_distro['perc_sum'] = german_jodel_shib_distro['shibboleths: lexical'] + german_jodel_shib_distro['shibboleths: other'] + german_jodel_shib_distro['non shibboleths']
german_jodel_shib_distro.loc['average'] = round(german_jodel_shib_distro.mean(axis=0), 1)

german_jodel_shib_distro

,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum
file,,,,,,,,,
list1,57.0,54.0,289.0,400.0,400.0,14.2,13.5,72.2,99.9
list2,20.0,69.0,311.0,400.0,400.0,5.0,17.2,77.8,100.0
list3,44.0,57.0,299.0,400.0,400.0,11.0,14.2,74.8,100.0
list4,23.0,50.0,327.0,400.0,400.0,5.8,12.5,81.8,100.1
list5,19.0,59.0,322.0,400.0,400.0,4.8,14.8,80.5,100.1
list6,40.0,59.0,301.0,400.0,400.0,10.0,14.8,75.2,100.0
list7,29.0,60.0,311.0,400.0,400.0,7.2,15.0,77.8,100.0
list8,21.0,42.0,337.0,400.0,400.0,5.2,10.5,84.2,99.9
average,31.6,56.2,312.1,400.0,400.0,7.9,14.1,78.0,100.0


In [29]:
german_jodel_shib_distro = pd.concat([german_jodel_shib_distro, german_jodel_methods_df], axis=1)
german_jodel_shib_distro

,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum,method
list1,57.0,54.0,289.0,400.0,400.0,14.2,13.5,72.2,99.9,ig_0.1
list2,20.0,69.0,311.0,400.0,400.0,5.0,17.2,77.8,100.0,ig_0.6
list3,44.0,57.0,299.0,400.0,400.0,11.0,14.2,74.8,100.0,lime_0.1
list4,23.0,50.0,327.0,400.0,400.0,5.8,12.5,81.8,100.1,lime_0.6
list5,19.0,59.0,322.0,400.0,400.0,4.8,14.8,80.5,100.1,shap_0.6
list6,40.0,59.0,301.0,400.0,400.0,10.0,14.8,75.2,100.0,shap_0.1
list7,29.0,60.0,311.0,400.0,400.0,7.2,15.0,77.8,100.0,loo_0.1
list8,21.0,42.0,337.0,400.0,400.0,5.2,10.5,84.2,99.9,loo_0.6
average,31.6,56.2,312.1,400.0,400.0,7.9,14.1,78.0,100.0,average


In [30]:
german_jodel_shib_distro = german_jodel_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
german_jodel_shib_distro

,list,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum
method,,,,,,,,,,
average,average,31.6,56.2,312.1,400.0,400.0,7.9,14.1,78.0,100.0
ig_0.1,list1,57.0,54.0,289.0,400.0,400.0,14.2,13.5,72.2,99.9
ig_0.6,list2,20.0,69.0,311.0,400.0,400.0,5.0,17.2,77.8,100.0
lime_0.1,list3,44.0,57.0,299.0,400.0,400.0,11.0,14.2,74.8,100.0
lime_0.6,list4,23.0,50.0,327.0,400.0,400.0,5.8,12.5,81.8,100.1
loo_0.1,list7,29.0,60.0,311.0,400.0,400.0,7.2,15.0,77.8,100.0
loo_0.6,list8,21.0,42.0,337.0,400.0,400.0,5.2,10.5,84.2,99.9
shap_0.1,list6,40.0,59.0,301.0,400.0,400.0,10.0,14.8,75.2,100.0
shap_0.6,list5,19.0,59.0,322.0,400.0,400.0,4.8,14.8,80.5,100.1


In [31]:
german_jodel_overview = german_jodel_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
german_jodel_overview.to_csv('tables/german_jodel_overview.csv')
german_jodel_overview

,shibboleths: lexical,shibboleths: other,non shibboleths
method,,,
average,7.9,14.1,78.0
ig_0.1,14.2,13.5,72.2
ig_0.6,5.0,17.2,77.8
lime_0.1,11.0,14.2,74.8
lime_0.6,5.8,12.5,81.8
loo_0.1,7.2,15.0,77.8
loo_0.6,5.2,10.5,84.2
shap_0.1,10.0,14.8,75.2
shap_0.6,4.8,14.8,80.5


### greek_dialects

In [32]:
greek_dialects_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.greek.txt')
greek_dialects_methods_df = pd.DataFrame.from_dict(greek_dialects_methods, orient='index', columns=['method'])
greek_dialects_methods_df

,method
average,average
list1,ig_0.6
list2,shap_0.1
list3,loo_0.1
list4,loo_0.6
list5,shap_0.6
list6,ig_0.1
list7,lime_0.6
list8,lime_0.1


In [33]:
greek_dialects_df = make_dataset_df('../../manual_eval/greek_dialects')
greek_dialects_df

Skip file: OLD


,token,label,score,selection_freq,feature,file
0,τσίτα-να!-οψάργας,CRE,1.477226,0.10,Lex,list6
1,κι’αλλ’αγάπη,CRE,1.383781,0.10,Unmarked,list6
2,ν-αναστενάζεις,CRE,1.382307,0.10,Unmarked,list6
3,νταμπακοπαφλάγονας,CRE,1.368082,0.10,Unmarked,list6
4,τσ’αφανίσω»,CRE,1.327946,0.10,Morph,list6
...,...,...,...,...,...,...
3195,φρουροί,PON,0.373081,0.24,Unmarked,list8
3196,καρενέσε,PON,0.370964,0.10,Unmarked,list8
3197,μεξικού,PON,0.369579,0.96,NE,list8
3198,χλωρίζνε,PON,0.366267,0.10,Morph,list8


In [34]:
greek_dialects_df[greek_dialects_df['file'] ==  'list2']

,token,label,score,selection_freq,feature,file
2400,θυμάται_στο,CRE,1.024675,0.10,Unmarked,list2
2401,μετακινήσω,CRE,1.009906,0.10,Unmarked,list2
2402,ακροατή/αναγνώστη,CRE,1.003343,0.10,Unmarked,list2
2403,ξεφωνήση,CRE,0.999911,0.10,Morph,list2
2404,ξεπρήσθη,CRE,0.999860,0.10,Morph,list2
...,...,...,...,...,...,...
2795,τιούτσεβα,PON,0.213957,0.25,NE,list2
2796,επεστάθεν,PON,0.212394,0.34,Lex,list2
2797,ασατήν,PON,0.212127,0.10,Lex,list2
2798,οροπέδιο,PON,0.211999,0.10,Unmarked,list2


In [35]:
greek_dialects_df.value_counts('feature')

feature
Unmarked    1516
Phon         604
NE           420
Lex          342
Morph        312
Other          3
I              1
ΝΕ             1
Spell          1
Name: count, dtype: int64

In [36]:
greek_dialects_nannots = get_number_annots(greek_dialects_df)
greek_dialects_nannots

,total
file,
list6,400
list7,400
list5,400
list4,400
list1,400
list3,400
list2,400
list8,400


In [37]:
greek_dialects_shib_distro = greek_dialects_df[greek_dialects_df['feature'] == 'Lex'].groupby('file').size().to_frame(name='lex shib')
greek_dialects_shib_distro['nonlex shib'] = greek_dialects_df[(greek_dialects_df['feature'] != 'NE') & (greek_dialects_df['feature'] != 'Unmarked') & (greek_dialects_df['feature'] != 'Lex')].groupby('file').size()
greek_dialects_shib_distro['non shib'] = greek_dialects_df[(greek_dialects_df['feature'] == 'NE') | (greek_dialects_df['feature'] == 'Unmarked')].groupby('file').size()
greek_dialects_shib_distro.loc[:, 'sum'] = greek_dialects_shib_distro.sum(axis=1)
greek_dialects_shib_distro = pd.concat([greek_dialects_shib_distro, greek_dialects_nannots], axis=1)
greek_dialects_shib_distro['shibboleths: lexical'] = round(greek_dialects_shib_distro['lex shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['shibboleths: other'] = round(greek_dialects_shib_distro['nonlex shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['non shibboleths'] = round(greek_dialects_shib_distro['non shib'] / \
                                                          greek_dialects_shib_distro['total'] * 100, 1)
greek_dialects_shib_distro['perc_sum'] = greek_dialects_shib_distro['shibboleths: lexical'] + greek_dialects_shib_distro['shibboleths: other'] \
+ greek_dialects_shib_distro['non shibboleths']
greek_dialects_shib_distro.loc['average'] = round(greek_dialects_shib_distro.mean(axis=0) , 1)

greek_dialects_shib_distro

,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum
file,,,,,,,,,
list1,74.0,164.0,162.0,400.0,400.0,18.5,41.0,40.5,100.0
list2,38.0,122.0,240.0,400.0,400.0,9.5,30.5,60.0,100.0
list3,25.0,80.0,295.0,400.0,400.0,6.2,20.0,73.8,100.0
list4,30.0,72.0,298.0,400.0,400.0,7.5,18.0,74.5,100.0
list5,30.0,154.0,216.0,400.0,400.0,7.5,38.5,54.0,100.0
list6,72.0,108.0,220.0,400.0,400.0,18.0,27.0,55.0,100.0
list7,31.0,94.0,275.0,400.0,400.0,7.8,23.5,68.8,100.1
list8,42.0,128.0,230.0,400.0,400.0,10.5,32.0,57.5,100.0
average,42.8,115.2,242.0,400.0,400.0,10.7,28.8,60.5,100.0


In [38]:
greek_dialects_shib_distro = pd.concat([greek_dialects_shib_distro, greek_dialects_methods_df], axis=1)
greek_dialects_shib_distro

,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum,method
list1,74.0,164.0,162.0,400.0,400.0,18.5,41.0,40.5,100.0,ig_0.6
list2,38.0,122.0,240.0,400.0,400.0,9.5,30.5,60.0,100.0,shap_0.1
list3,25.0,80.0,295.0,400.0,400.0,6.2,20.0,73.8,100.0,loo_0.1
list4,30.0,72.0,298.0,400.0,400.0,7.5,18.0,74.5,100.0,loo_0.6
list5,30.0,154.0,216.0,400.0,400.0,7.5,38.5,54.0,100.0,shap_0.6
list6,72.0,108.0,220.0,400.0,400.0,18.0,27.0,55.0,100.0,ig_0.1
list7,31.0,94.0,275.0,400.0,400.0,7.8,23.5,68.8,100.1,lime_0.6
list8,42.0,128.0,230.0,400.0,400.0,10.5,32.0,57.5,100.0,lime_0.1
average,42.8,115.2,242.0,400.0,400.0,10.7,28.8,60.5,100.0,average


In [39]:
greek_dialects_shib_distro = greek_dialects_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
greek_dialects_shib_distro

,list,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum
method,,,,,,,,,,
average,average,42.8,115.2,242.0,400.0,400.0,10.7,28.8,60.5,100.0
ig_0.1,list6,72.0,108.0,220.0,400.0,400.0,18.0,27.0,55.0,100.0
ig_0.6,list1,74.0,164.0,162.0,400.0,400.0,18.5,41.0,40.5,100.0
lime_0.1,list8,42.0,128.0,230.0,400.0,400.0,10.5,32.0,57.5,100.0
lime_0.6,list7,31.0,94.0,275.0,400.0,400.0,7.8,23.5,68.8,100.1
loo_0.1,list3,25.0,80.0,295.0,400.0,400.0,6.2,20.0,73.8,100.0
loo_0.6,list4,30.0,72.0,298.0,400.0,400.0,7.5,18.0,74.5,100.0
shap_0.1,list2,38.0,122.0,240.0,400.0,400.0,9.5,30.5,60.0,100.0
shap_0.6,list5,30.0,154.0,216.0,400.0,400.0,7.5,38.5,54.0,100.0


In [40]:
greek_dialects_overview = greek_dialects_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
greek_dialects_overview.to_csv('tables/greek_dialects_overview.csv')
greek_dialects_overview

,shibboleths: lexical,shibboleths: other,non shibboleths
method,,,
average,10.7,28.8,60.5
ig_0.1,18.0,27.0,55.0
ig_0.6,18.5,41.0,40.5
lime_0.1,10.5,32.0,57.5
lime_0.6,7.8,23.5,68.8
loo_0.1,6.2,20.0,73.8
loo_0.6,7.5,18.0,74.5
shap_0.1,9.5,30.5,60.0
shap_0.6,7.5,38.5,54.0


### finnish_suomi24

In [41]:
finnish_suomi24_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.finnish.txt')
finnish_suomi24_methods_df = pd.DataFrame.from_dict(finnish_suomi24_methods, orient='index', columns=['method'])
finnish_suomi24_methods_df

,method
average,average
list1,shap_0.6
list2,loo_0.6
list3,ig_0.6
list4,loo_0.1
list5,lime_0.6
list6,lime_0.1
list7,ig_0.1
list8,shap_0.1


In [42]:
finnish_suomi24_df = make_dataset_df('../../manual_eval/finnish')
finnish_suomi24_df

,Unnamed: 0,token,label,score,selection_freq,category,lex_other_not,file
0,0,täötyy,east,0.863224,0.101010,phon,other,list8
1,1,pahimoelleen,east,0.847876,0.101010,phon,other,list8
2,2,sullakii,east,0.826737,0.202020,morph,other,list8
3,3,paekkaan,east,0.811686,0.101010,phon,other,list8
4,4,piästii,east,0.804127,0.101010,phon,other,list8
...,...,...,...,...,...,...,...,...
2015,295,varovainen,west,0.258827,0.101010,not,not,list7
2016,296,miä,west,0.258084,0.161616,lex,lex,list7
2017,297,nää,west,0.257057,0.191919,not,not,list7
2018,298,justihin,west,0.256982,0.222222,morph,other,list7


In [43]:
finnish_suomi24_df.value_counts('label')

label
north    680
east     673
west     667
Name: count, dtype: int64

In [44]:
finnish_suomi24_nannots = get_number_annots(finnish_suomi24_df)
finnish_suomi24_nannots

,total
file,
list8,300
list4,300
list6,300
list7,300
list5,221
list2,207
list1,206
list3,186


In [45]:
finnish_suomi24_shib_distro = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'lex'].groupby('file').size().to_frame(name='lex shib')
finnish_suomi24_shib_distro['nonlex shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] == 'other'].groupby('file').size()
finnish_suomi24_shib_distro['non shib'] = finnish_suomi24_df[finnish_suomi24_df['lex_other_not'] ==  'not'].groupby('file').size()
finnish_suomi24_shib_distro.loc[:, 'sum'] = finnish_suomi24_shib_distro.sum(axis=1)
finnish_suomi24_shib_distro = pd.concat([finnish_suomi24_shib_distro, finnish_suomi24_nannots], axis=1)
finnish_suomi24_shib_distro['shibboleths: lexical'] = round(finnish_suomi24_shib_distro['lex shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['shibboleths: other'] = round(finnish_suomi24_shib_distro['nonlex shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['non shibboleths'] = round(finnish_suomi24_shib_distro['non shib'] / 
                                                             finnish_suomi24_shib_distro['total'] * 100, 1)
finnish_suomi24_shib_distro['perc_sum'] = finnish_suomi24_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']].sum(axis=1)
finnish_suomi24_shib_distro.loc['average'] = round(finnish_suomi24_shib_distro.mean(axis=0), 1)
finnish_suomi24_shib_distro

,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum
file,,,,,,,,,
list1,19.0,49.0,138.0,206.0,206.0,9.2,23.8,67.0,100.0
list2,21.0,44.0,142.0,207.0,207.0,10.1,21.3,68.6,100.0
list3,19.0,42.0,125.0,186.0,186.0,10.2,22.6,67.2,100.0
list4,17.0,196.0,87.0,300.0,300.0,5.7,65.3,29.0,100.0
list5,21.0,55.0,145.0,221.0,221.0,9.5,24.9,65.6,100.0
list6,21.0,201.0,78.0,300.0,300.0,7.0,67.0,26.0,100.0
list7,9.0,186.0,105.0,300.0,300.0,3.0,62.0,35.0,100.0
list8,16.0,206.0,78.0,300.0,300.0,5.3,68.7,26.0,100.0
average,17.9,122.4,112.2,252.5,252.5,7.5,44.4,48.0,100.0


In [46]:
finnish_suomi24_shib_distro = pd.concat([finnish_suomi24_shib_distro, finnish_suomi24_methods_df], axis=1)
finnish_suomi24_shib_distro

,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum,method
list1,19.0,49.0,138.0,206.0,206.0,9.2,23.8,67.0,100.0,shap_0.6
list2,21.0,44.0,142.0,207.0,207.0,10.1,21.3,68.6,100.0,loo_0.6
list3,19.0,42.0,125.0,186.0,186.0,10.2,22.6,67.2,100.0,ig_0.6
list4,17.0,196.0,87.0,300.0,300.0,5.7,65.3,29.0,100.0,loo_0.1
list5,21.0,55.0,145.0,221.0,221.0,9.5,24.9,65.6,100.0,lime_0.6
list6,21.0,201.0,78.0,300.0,300.0,7.0,67.0,26.0,100.0,lime_0.1
list7,9.0,186.0,105.0,300.0,300.0,3.0,62.0,35.0,100.0,ig_0.1
list8,16.0,206.0,78.0,300.0,300.0,5.3,68.7,26.0,100.0,shap_0.1
average,17.9,122.4,112.2,252.5,252.5,7.5,44.4,48.0,100.0,average


In [47]:
finnish_suomi24_shib_distro = finnish_suomi24_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
finnish_suomi24_shib_distro

,list,lex shib,nonlex shib,non shib,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,perc_sum
method,,,,,,,,,,
average,average,17.9,122.4,112.2,252.5,252.5,7.5,44.4,48.0,100.0
ig_0.1,list7,9.0,186.0,105.0,300.0,300.0,3.0,62.0,35.0,100.0
ig_0.6,list3,19.0,42.0,125.0,186.0,186.0,10.2,22.6,67.2,100.0
lime_0.1,list6,21.0,201.0,78.0,300.0,300.0,7.0,67.0,26.0,100.0
lime_0.6,list5,21.0,55.0,145.0,221.0,221.0,9.5,24.9,65.6,100.0
loo_0.1,list4,17.0,196.0,87.0,300.0,300.0,5.7,65.3,29.0,100.0
loo_0.6,list2,21.0,44.0,142.0,207.0,207.0,10.1,21.3,68.6,100.0
shap_0.1,list8,16.0,206.0,78.0,300.0,300.0,5.3,68.7,26.0,100.0
shap_0.6,list1,19.0,49.0,138.0,206.0,206.0,9.2,23.8,67.0,100.0


In [48]:
finnish_suomi24_overview = finnish_suomi24_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths']]
finnish_suomi24_overview.to_csv('tables/finnish_suomi24_overview.csv')
finnish_suomi24_overview

,shibboleths: lexical,shibboleths: other,non shibboleths
method,,,
average,7.5,44.4,48.0
ig_0.1,3.0,62.0,35.0
ig_0.6,10.2,22.6,67.2
lime_0.1,7.0,67.0,26.0
lime_0.6,9.5,24.9,65.6
loo_0.1,5.7,65.3,29.0
loo_0.6,10.1,21.3,68.6
shap_0.1,5.3,68.7,26.0
shap_0.6,9.2,23.8,67.0


### estonian_voro

In [49]:
estonian_voro_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.estonian_voro.txt')
estonian_voro_methods_df = pd.DataFrame.from_dict(estonian_voro_methods, orient='index', columns=['method'])
estonian_voro_methods_df

,method
average,average
list1,ig_0.6
list2,ig_0.1
list3,loo_0.1
list4,shap_0.6
list5,shap_0.1
list6,loo_0.6
list7,lime_0.6
list8,lime_0.1


In [50]:
estonian_voro_df = make_dataset_df('../../manual_eval/eesti-voro')
estonian_voro_df

Skip file: evaluateLists.py


,token,base form,cognate,label,Phon-Morph-infl-Morph-der-Spelling,Lexicon,NE,Other,Unmarked,Unknown,Comment,file
0,jookseb,jooksma,ju̬u̬skma,est,x,NaN,NaN,NaN,NaN,NaN,NaN,list4
1,jooksis,jooksma,ju̬u̬skma,est,x,NaN,NaN,NaN,NaN,NaN,NaN,list4
2,veab,vedama,vidämä,est,xx,NaN,NaN,NaN,NaN,NaN,NaN,list4
3,tõmbab,tõmbama,tõ̭mbama,est,x,NaN,NaN,NaN,NaN,NaN,NaN,list4
4,õunad,õun,!=? upin,est,x,x,NaN,NaN,NaN,NaN,NaN,list4
...,...,...,...,...,...,...,...,...,...,...,...,...
1595,vöie,vüü,vöö,vro,x,NaN,NaN,NaN,NaN,NaN,NaN,list2
1596,käänüskõrd,käänüskõrd,!= paradigma,vro,NaN,x,NaN,NaN,NaN,NaN,NaN,list2
1597,sainapapõr,sainapapõŕ,!= tapeet,vro,NaN,x,NaN,NaN,NaN,NaN,NaN,list2
1598,paljodõn,palľo,palju,vro,x,NaN,NaN,NaN,NaN,NaN,NaN,list2


In [51]:
estonian_voro_nannots = get_number_annots(estonian_voro_df)
estonian_voro_nannots

,total
file,
list4,200
list3,200
list7,200
list8,200
list1,200
list5,200
list6,200
list2,200


In [52]:
estonian_voro_df[estonian_voro_df['Lexicon'].str.contains(r'x|\?', na=False)]

,token,base form,cognate,label,Phon-Morph-infl-Morph-der-Spelling,Lexicon,NE,Other,Unmarked,Unknown,Comment,file
4,õunad,õun,!=? upin,est,x,x,NaN,NaN,NaN,NaN,NaN,list4
8,tõusis,tõusma,!= nõsõma,est,NaN,x,NaN,NaN,NaN,NaN,NaN,list4
9,tõuseb,tõusma,!= nõsõma,est,x,x,NaN,NaN,NaN,NaN,NaN,list4
11,vaikselt,vaikselt,!= vaikligult,est,x,x,NaN,NaN,NaN,NaN,derivation,list4
24,kiiremini,kiiremini,!= kipõmbahe,est,NaN,x,NaN,NaN,NaN,NaN,NaN,list4
...,...,...,...,...,...,...,...,...,...,...,...,...
1584,korgõrõuhkus,korgõrõuhkus,!=? kõrgrõhk,vro,NaN,x,NaN,NaN,NaN,NaN,NaN,list2
1589,teedüstahvli,teedüstahvli (= teedüssetahvli?),(teadetetahvel?),vro,NaN,x,NaN,NaN,NaN,NaN,derivation,list2
1593,valmispannusõgaq,valmispannus,!= preparaat (a different dictionary also know...,vro,NaN,x,NaN,NaN,NaN,NaN,derivation,list2
1596,käänüskõrd,käänüskõrd,!= paradigma,vro,NaN,x,NaN,NaN,NaN,NaN,NaN,list2


In [53]:
estonian_voro_df["unk"] = 0
estonian_voro_df.loc[(estonian_voro_df['Unknown'].str.contains('x', na=False) | estonian_voro_df['Other'].str.contains('x', na=False)), "unk"] = 1
estonian_voro_df["noshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & estonian_voro_df['Unmarked'].str.contains('x', na=False)), "noshib"] = 1
estonian_voro_df["lexshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & (estonian_voro_df["noshib"] == 0) & estonian_voro_df['Lexicon'].str.contains('x', na=False)), "lexshib"] = 1
estonian_voro_df["nonlexshib"] = 0
estonian_voro_df.loc[((estonian_voro_df["unk"] == 0) & (estonian_voro_df["noshib"] == 0) & (estonian_voro_df["lexshib"] == 0) & estonian_voro_df['Phon-Morph-infl-Morph-der-Spelling'].str.contains(r'x|s', na=False)), "nonlexshib"] = 1

estonian_voro_df.loc[estonian_voro_df["unk"] + estonian_voro_df["noshib"] + estonian_voro_df["lexshib"] + estonian_voro_df["nonlexshib"] == 0, "unk"] += 1

estonian_voro_df[estonian_voro_df["unk"] >= 1]

,token,base form,cognate,label,Phon-Morph-infl-Morph-der-Spelling,Lexicon,NE,Other,Unmarked,Unknown,Comment,file,unk,noshib,lexshib,nonlexshib
107,nal,? tokenisation mistakenly splitting nal´alt?,?,vro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,list4,1,0,0,0
206,kaebu,kaebu(-punn),"kitupunn, kaivatś",est,NaN,x,NaN,x,NaN,NaN,tokenisation problem,list3,1,0,0,0
267,se,?,?,est,NaN,NaN,NaN,x,NaN,NaN,virhe,list3,1,0,0,0
298,edas,edas-,iin-,est,x,NaN,NaN,x,NaN,NaN,tokenisation problem,list3,1,0,0,0
301,passmisõst,?,?,vro,x,NaN,NaN,NaN,NaN,x,NaN,list3,1,0,0,0
321,innõ,ynnõ,ainult,vro,x,x,NaN,NaN,NaN,x,NaN,list3,1,0,0,0
356,mustasitikõ,?,?,vro,x,NaN,NaN,NaN,NaN,x,NaN,list3,1,0,0,0
364,ülekuvvõkümne,ülekuvvõkümne,üle kuuekümne,vro,x,NaN,NaN,x,NaN,NaN,virhe?,list3,1,0,0,0
392,hinnästilmarahva,hinnäst ilmarahvas,ennast != külä-/linnarahvas,vro,NaN,x,NaN,x,NaN,NaN,"virhe? ehkä ""hinnäst ilmarahva""",list3,1,0,0,0
473,pandi,panema,pandma,est,NaN,NaN,NaN,x,NaN,NaN,NaN,list7,1,0,0,0


In [54]:
estonian_voro_shib_distro = estonian_voro_df[estonian_voro_df['lexshib']==1].groupby('file').size().to_frame(name="lex shib")
estonian_voro_shib_distro['non lex shib'] = estonian_voro_df[estonian_voro_df['nonlexshib']==1].groupby('file').size()
estonian_voro_shib_distro['non shib'] = estonian_voro_df[estonian_voro_df['noshib']==1].groupby('file').size()
estonian_voro_shib_distro['rest'] = estonian_voro_df[estonian_voro_df['unk']==1].groupby('file').size()
estonian_voro_shib_distro.loc[:, 'sum'] = estonian_voro_shib_distro.sum(axis=1)

estonian_voro_shib_distro = pd.concat([estonian_voro_shib_distro, estonian_voro_nannots], axis=1)
estonian_voro_shib_distro['shibboleths: lexical'] = round(estonian_voro_shib_distro['lex shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['shibboleths: other'] = round(estonian_voro_shib_distro['non lex shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['non shibboleths'] = round(estonian_voro_shib_distro['non shib'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)
estonian_voro_shib_distro['other'] = round(estonian_voro_shib_distro['rest'] \
                                                               / estonian_voro_shib_distro['total'] * 100, 1)

estonian_voro_shib_distro.loc[:, 'perc_sum'] = estonian_voro_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']].sum(axis=1)
estonian_voro_shib_distro.loc['average'] = round(estonian_voro_shib_distro.mean(axis=0), 1)

estonian_voro_shib_distro

,lex shib,non lex shib,non shib,rest,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc_sum
file,,,,,,,,,,,
list1,35.0,147.0,16.0,2.0,200.0,200.0,17.5,73.5,8.0,1.0,100.0
list2,72.0,114.0,7.0,7.0,200.0,200.0,36.0,57.0,3.5,3.5,100.0
list3,52.0,133.0,7.0,8.0,200.0,200.0,26.0,66.5,3.5,4.0,100.0
list4,22.0,153.0,24.0,1.0,200.0,200.0,11.0,76.5,12.0,0.5,100.0
list5,54.0,136.0,4.0,6.0,200.0,200.0,27.0,68.0,2.0,3.0,100.0
list6,18.0,141.0,39.0,2.0,200.0,200.0,9.0,70.5,19.5,1.0,100.0
list7,17.0,140.0,42.0,1.0,200.0,200.0,8.5,70.0,21.0,0.5,100.0
list8,55.0,134.0,2.0,9.0,200.0,200.0,27.5,67.0,1.0,4.5,100.0
average,40.6,137.2,17.6,4.5,200.0,200.0,20.3,68.6,8.8,2.2,100.0


In [55]:
estonian_voro_shib_distro = pd.concat([estonian_voro_shib_distro, estonian_voro_methods_df], axis=1)
estonian_voro_shib_distro

,lex shib,non lex shib,non shib,rest,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc_sum,method
list1,35.0,147.0,16.0,2.0,200.0,200.0,17.5,73.5,8.0,1.0,100.0,ig_0.6
list2,72.0,114.0,7.0,7.0,200.0,200.0,36.0,57.0,3.5,3.5,100.0,ig_0.1
list3,52.0,133.0,7.0,8.0,200.0,200.0,26.0,66.5,3.5,4.0,100.0,loo_0.1
list4,22.0,153.0,24.0,1.0,200.0,200.0,11.0,76.5,12.0,0.5,100.0,shap_0.6
list5,54.0,136.0,4.0,6.0,200.0,200.0,27.0,68.0,2.0,3.0,100.0,shap_0.1
list6,18.0,141.0,39.0,2.0,200.0,200.0,9.0,70.5,19.5,1.0,100.0,loo_0.6
list7,17.0,140.0,42.0,1.0,200.0,200.0,8.5,70.0,21.0,0.5,100.0,lime_0.6
list8,55.0,134.0,2.0,9.0,200.0,200.0,27.5,67.0,1.0,4.5,100.0,lime_0.1
average,40.6,137.2,17.6,4.5,200.0,200.0,20.3,68.6,8.8,2.2,100.0,average


In [56]:
estonian_voro_shib_distro = estonian_voro_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
estonian_voro_shib_distro

,list,lex shib,non lex shib,non shib,rest,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc_sum
method,,,,,,,,,,,,
average,average,40.6,137.2,17.6,4.5,200.0,200.0,20.3,68.6,8.8,2.2,100.0
ig_0.1,list2,72.0,114.0,7.0,7.0,200.0,200.0,36.0,57.0,3.5,3.5,100.0
ig_0.6,list1,35.0,147.0,16.0,2.0,200.0,200.0,17.5,73.5,8.0,1.0,100.0
lime_0.1,list8,55.0,134.0,2.0,9.0,200.0,200.0,27.5,67.0,1.0,4.5,100.0
lime_0.6,list7,17.0,140.0,42.0,1.0,200.0,200.0,8.5,70.0,21.0,0.5,100.0
loo_0.1,list3,52.0,133.0,7.0,8.0,200.0,200.0,26.0,66.5,3.5,4.0,100.0
loo_0.6,list6,18.0,141.0,39.0,2.0,200.0,200.0,9.0,70.5,19.5,1.0,100.0
shap_0.1,list5,54.0,136.0,4.0,6.0,200.0,200.0,27.0,68.0,2.0,3.0,100.0
shap_0.6,list4,22.0,153.0,24.0,1.0,200.0,200.0,11.0,76.5,12.0,0.5,100.0


In [57]:
estonian_voro_overview = estonian_voro_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
estonian_voro_overview.to_csv('tables/estonian_voro_overview.csv')
estonian_voro_overview

,shibboleths: lexical,shibboleths: other,non shibboleths,other
method,,,,
average,20.3,68.6,8.8,2.2
ig_0.1,36.0,57.0,3.5,3.5
ig_0.6,17.5,73.5,8.0,1.0
lime_0.1,27.5,67.0,1.0,4.5
lime_0.6,8.5,70.0,21.0,0.5
loo_0.1,26.0,66.5,3.5,4.0
loo_0.6,9.0,70.5,19.5,1.0
shap_0.1,27.0,68.0,2.0,3.0
shap_0.6,11.0,76.5,12.0,0.5


### scandinavian_slide

In [58]:
scandinavian_slide_methods = get_method_dict('../../manual_eval/randomized_lists/randomized_lists.scandinavian.txt')
scandinavian_slide_methods_df = pd.DataFrame.from_dict(scandinavian_slide_methods, orient='index', columns=['method'])
scandinavian_slide_methods_df

,method
average,average
list1,lime_0.1
list2,ig_0.1
list3,shap_0.1
list4,shap_0.6
list5,loo_0.1
list6,loo_0.6
list7,ig_0.6
list8,lime_0.6


In [59]:
scandinavian_slide_df = make_dataset_df('../../manual_eval/scandinavian')
scandinavian_slide_df

Skip file: pyenv
Skip file: countBadEx.py
Skip file: scandinavian_unique_annotated.csv
Skip file: transferAnnotations.py
Skip file: orig
Skip file: countMainCategories.py
Skip file: evaluateLists.py
Skip file: namedEntities.py
Skip file: scandinavian_unique.txt


,id,token,label,score,selection_freq,in_blacklist,cognate_da,cognate_nb,cognate_nn,cognate_sv,NotShib,NotMinimal,Morph,Phon_Spell,NE,Typo_Unk,LexShib,Counterex,NonLexShib,file
0,0,ydmyghed,da,1.008674,0.10,False,NaN,ydmykhet,audmykt,ödmjukhet,False,False,True,True,False,False,False,False,True,list3
1,1,styrtregnede,da,1.000334,0.10,False,NaN,styrtregnet,styrtregna,NaN,False,False,True,False,False,False,False,False,True,list3
2,2,broderede,da,1.000113,0.10,False,NaN,broderte,NaN,broderade,False,False,True,False,False,False,False,False,True,list3
3,3,åbningsudsalg,da,0.999946,0.10,False,NaN,åpningssalg?,NaN,öppningsrea?,False,False,False,False,False,False,True,False,False,list3
4,4,orddelingsfejl,da,0.999925,0.10,False,NaN,orddelingsfeil,NaN,NaN,False,False,False,True,False,False,False,False,True,list3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3195,395,lämnade,sv,0.155722,0.98,False,efterlod,etterlot,NaN,NaN,False,False,False,False,False,False,True,False,False,list8
3196,396,färdiga,sv,0.154972,0.75,False,NaN,ferdige,ferdige,NaN,False,False,False,True,False,False,False,False,True,list8
3197,397,väntar,sv,0.154830,1.00,False,venter,venter,ventar,NaN,False,False,True,True,False,False,False,False,True,list8
3198,398,förändras,sv,0.154672,0.77,False,NaN,forandre,forandre,NaN,False,False,True,True,False,False,False,False,True,list8


In [60]:
scandinavian_slide_df[(scandinavian_slide_df['NE'] == True) & (scandinavian_slide_df['NonLexShib'] == True)]

,id,token,label,score,selection_freq,in_blacklist,cognate_da,cognate_nb,cognate_nn,cognate_sv,NotShib,NotMinimal,Morph,Phon_Spell,NE,Typo_Unk,LexShib,Counterex,NonLexShib,file
131,131,bergenskjendiser,nb,0.843779,0.10,False,NaN,NaN,kjendisar,kändisar,False,True,False,True,True,False,False,False,True,list3
219,219,telemarkingane,nn,0.497270,0.10,False,NaN,telemarkingene,NaN,NaN,False,False,True,False,True,False,False,False,True,list3
237,237,sveitsar,nn,0.426922,0.10,False,schweizer,sveitser,NaN,schweizare,False,False,True,True,True,False,False,False,True,list3
248,248,noregscupfinalen,nn,0.405608,0.10,False,NaN,norges,NaN,NaN,False,True,False,False,True,False,False,False,True,list3
252,252,seljordingar,nn,0.389857,0.10,False,NaN,seljordinger,NaN,NaN,False,False,True,False,True,False,False,False,True,list3
281,281,seljordingane,nn,0.320533,0.10,False,NaN,seljordingene,NaN,NaN,False,False,True,False,True,False,False,False,True,list3
509,109,bergenskjendiser,nb,1.239547,0.10,False,NaN,NaN,kjendisar,kändisar,False,True,False,True,True,False,False,False,True,list2
538,138,spellemannpriser,nb,1.023061,0.10,False,NaN,NaN,spellemannprisar,NaN,False,True,True,False,True,False,False,False,True,list2
571,171,skottene,nb,0.947544,0.11,False,skotterne,NaN,skottane,NaN,False,False,True,False,True,False,False,False,True,list2
581,181,ffk-fansens,nb,0.920009,0.10,False,NaN,NaN,fansen,fanen,False,True,True,False,True,False,False,False,True,list2


In [61]:
scandinavian_slide_nannots = get_number_annots(scandinavian_slide_df)
scandinavian_slide_nannots

,total
file,
list3,400
list2,400
list1,400
list5,400
list4,400
list6,400
list7,400
list8,400


In [62]:
scandinavian_slide_shib_distro = scandinavian_slide_df[scandinavian_slide_df['LexShib'] == True].groupby('file').size().to_frame(name='lex shib')
scandinavian_slide_shib_distro['nonlex shib'] = scandinavian_slide_df[scandinavian_slide_df['NonLexShib'] == True].groupby('file').size()
scandinavian_slide_shib_distro['non shib'] = scandinavian_slide_df[(scandinavian_slide_df['NotShib'] ==  True)].groupby('file').size()
scandinavian_slide_shib_distro['rest'] = scandinavian_slide_df[(scandinavian_slide_df['Typo_Unk'] == True)].groupby('file').size()
scandinavian_slide_shib_distro.loc[:, 'sum'] = scandinavian_slide_shib_distro.sum(axis=1)
scandinavian_slide_shib_distro = pd.concat([scandinavian_slide_shib_distro, scandinavian_slide_nannots], axis=1)

scandinavian_slide_shib_distro['shibboleths: lexical'] = round(scandinavian_slide_shib_distro['lex shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['shibboleths: other'] = round(scandinavian_slide_shib_distro['nonlex shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['non shibboleths'] = round(scandinavian_slide_shib_distro['non shib'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)
scandinavian_slide_shib_distro['other'] = round(scandinavian_slide_shib_distro['rest'] \
                                                               / scandinavian_slide_shib_distro['total'] * 100, 1)

scandinavian_slide_shib_distro.loc[:, 'perc_sum'] = scandinavian_slide_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']].sum(axis=1)

scandinavian_slide_shib_distro.loc['average'] = round(scandinavian_slide_shib_distro.mean(axis=0), 1)

scandinavian_slide_shib_distro

,lex shib,nonlex shib,non shib,rest,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc_sum
file,,,,,,,,,,,
list1,77.0,217.0,93.0,13.0,400.0,400.0,19.2,54.2,23.2,3.2,99.8
list2,97.0,205.0,76.0,22.0,400.0,400.0,24.2,51.2,19.0,5.5,99.9
list3,95.0,222.0,71.0,12.0,400.0,400.0,23.8,55.5,17.8,3.0,100.1
list4,67.0,217.0,116.0,NaN,400.0,400.0,16.8,54.2,29.0,NaN,100.0
list5,77.0,213.0,99.0,11.0,400.0,400.0,19.2,53.2,24.8,2.8,100.0
list6,66.0,196.0,138.0,NaN,400.0,400.0,16.5,49.0,34.5,NaN,100.0
list7,61.0,215.0,124.0,NaN,400.0,400.0,15.2,53.8,31.0,NaN,100.0
list8,63.0,209.0,127.0,1.0,400.0,400.0,15.8,52.2,31.8,0.2,100.0
average,75.4,211.8,105.5,11.8,400.0,400.0,18.8,52.9,26.4,2.9,100.0


In [63]:
scandinavian_slide_shib_distro = pd.concat([scandinavian_slide_shib_distro, scandinavian_slide_methods_df], axis=1)
scandinavian_slide_shib_distro

,lex shib,nonlex shib,non shib,rest,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc_sum,method
list1,77.0,217.0,93.0,13.0,400.0,400.0,19.2,54.2,23.2,3.2,99.8,lime_0.1
list2,97.0,205.0,76.0,22.0,400.0,400.0,24.2,51.2,19.0,5.5,99.9,ig_0.1
list3,95.0,222.0,71.0,12.0,400.0,400.0,23.8,55.5,17.8,3.0,100.1,shap_0.1
list4,67.0,217.0,116.0,NaN,400.0,400.0,16.8,54.2,29.0,NaN,100.0,shap_0.6
list5,77.0,213.0,99.0,11.0,400.0,400.0,19.2,53.2,24.8,2.8,100.0,loo_0.1
list6,66.0,196.0,138.0,NaN,400.0,400.0,16.5,49.0,34.5,NaN,100.0,loo_0.6
list7,61.0,215.0,124.0,NaN,400.0,400.0,15.2,53.8,31.0,NaN,100.0,ig_0.6
list8,63.0,209.0,127.0,1.0,400.0,400.0,15.8,52.2,31.8,0.2,100.0,lime_0.6
average,75.4,211.8,105.5,11.8,400.0,400.0,18.8,52.9,26.4,2.9,100.0,average


In [ ]:
scandinavian_slide_shib_distro = scandinavian_slide_shib_distro.sort_values(by=['method']).reset_index(names=['list']).set_index('method')
scandinavian_slide_shib_distro

,list,lex shib,nonlex shib,non shib,rest,sum,total,shibboleths: lexical,shibboleths: other,non shibboleths,other,perc_sum
method,,,,,,,,,,,,
average,average,75.4,211.8,105.5,11.8,400.0,400.0,18.8,52.9,26.4,2.9,100.0
ig_0.1,list2,97.0,205.0,76.0,22.0,400.0,400.0,24.2,51.2,19.0,5.5,99.9
ig_0.6,list7,61.0,215.0,124.0,NaN,400.0,400.0,15.2,53.8,31.0,NaN,100.0
lime_0.1,list1,77.0,217.0,93.0,13.0,400.0,400.0,19.2,54.2,23.2,3.2,99.8
lime_0.6,list8,63.0,209.0,127.0,1.0,400.0,400.0,15.8,52.2,31.8,0.2,100.0
loo_0.1,list5,77.0,213.0,99.0,11.0,400.0,400.0,19.2,53.2,24.8,2.8,100.0
loo_0.6,list6,66.0,196.0,138.0,NaN,400.0,400.0,16.5,49.0,34.5,NaN,100.0
shap_0.1,list3,95.0,222.0,71.0,12.0,400.0,400.0,23.8,55.5,17.8,3.0,100.1
shap_0.6,list4,67.0,217.0,116.0,NaN,400.0,400.0,16.8,54.2,29.0,NaN,100.0


In [65]:
scandinavian_slide_overview = scandinavian_slide_shib_distro[['shibboleths: lexical', 'shibboleths: other', 'non shibboleths', 'other']]
scandinavian_slide_overview.to_csv('tables/scandinavian_slide_overview.csv')
scandinavian_slide_overview

,shibboleths: lexical,shibboleths: other,non shibboleths,other
method,,,,
average,18.8,52.9,26.4,2.9
ig_0.1,24.2,51.2,19.0,5.5
ig_0.6,15.2,53.8,31.0,NaN
lime_0.1,19.2,54.2,23.2,3.2
lime_0.6,15.8,52.2,31.8,0.2
loo_0.1,19.2,53.2,24.8,2.8
loo_0.6,16.5,49.0,34.5,NaN
shap_0.1,23.8,55.5,17.8,3.0
shap_0.6,16.8,54.2,29.0,NaN


## General overview table

In [72]:
bcms_setimes_overview.loc[:, 'dataset'] = 'bcms_setimes'
bcms_twitter_overview.loc[:, 'dataset'] = 'bcms_twitter'
finnish_suomi24_overview.loc[:, 'dataset'] = 'finnish_suomi24'
german_jodel_overview.loc[:, 'dataset'] = 'german_jodel'
greek_dialects_overview.loc[:, 'dataset'] = 'greek_dialects'
estonian_voro_overview.loc[:, 'dataset'] = 'estonian_voro'
scandinavian_slide_overview.loc[:, 'dataset'] = 'scandinavian_slide'

all_overview = pd.concat([bcms_twitter_overview, bcms_setimes_overview, finnish_suomi24_overview, \
                         german_jodel_overview, greek_dialects_overview, scandinavian_slide_overview, estonian_voro_overview], axis=0).reset_index(names=['method'])
#all_overview['method'] = all_overview['method'].str.split("_", expand=True)[0].str.upper() + ", p=" + all_overview['method'].str.split("_", expand=True)[1]
all_overview['method'] = all_overview['method'].map({
	"ig_0.1": "IG, $p=0.1$",
	"ig_0.6": "IG, $p=0.6$",
	"lime_0.1": "LIME, $p=0.1$",
	"lime_0.6": "LIME, $p=0.6$",
	"loo_0.1": "LOO, $p=0.1$",
	"loo_0.6": "LOO, $p=0.6$",
	"shap_0.1": "SHAP, $p=0.1$",
	"shap_0.6": "SHAP, $p=0.6$",
	"average": "average"
})
all_overview["other"] = all_overview["other"].fillna(0)
all_overview.to_csv('tables/all_overview.csv')
all_overview

,method,shibboleths: lexical,shibboleths: other,non shibboleths,other,dataset
0,average,3.7,24.9,69.0,2.4,bcms_twitter
1,"IG, $p=0.1$",4.0,19.8,73.2,3.0,bcms_twitter
2,"IG, $p=0.6$",2.2,27.0,67.2,3.5,bcms_twitter
3,"LIME, $p=0.1$",5.2,28.2,64.8,1.8,bcms_twitter
4,"LIME, $p=0.6$",3.8,27.3,67.0,2.0,bcms_twitter
...,...,...,...,...,...,...
58,"LIME, $p=0.6$",8.5,70.0,21.0,0.5,estonian_voro
59,"LOO, $p=0.1$",26.0,66.5,3.5,4.0,estonian_voro
60,"LOO, $p=0.6$",9.0,70.5,19.5,1.0,estonian_voro
61,"SHAP, $p=0.1$",27.0,68.0,2.0,3.0,estonian_voro
